# 09_Trailing_Stop_Loss — 個股緊急避險：移動停損實驗

## 目的

進入 Phase 4：加入個股級別的風險控管防護網。

目前的策略依賴「大盤 0050」作群體保護，但在牛市期間，如果我們買到的某一檔股票（如特定電子股）**突然月中暴發重大警惕並連續跌停**，我們只能「抱著它」直到當月底換倉才賣。

透過引進「移動停損 (Trailing Stop Loss)」，只要股票從買進後的「近期相對高點」回落超過 X%，即使還沒到月底換倉日，我們也立刻斷尾求生賣掉換現金。

這個實驗室會掃描不同的停損容忍度（例如 10%、15%、20%、無停損），看看加上這張防護網，會不會把我們的 Max Drawdown 進一步向下拉低到低於 25%。

In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import vectorbt as vbt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 切換目錄
original_cwd = os.getcwd()
try:
    if os.path.basename(original_cwd) == "Research":
        os.chdir("..")

    print("Loading cached data from finmind_cache...")
    close = pd.read_pickle("finmind_cache/close_wide.pkl")
    close.index = pd.to_datetime(close.index)

    # 載入前幾個階段優化好的最終版 Weights 表
    # 這個 Weights 已經包含了 TOP_K=40, keep_top_k=80(Inertia), WEIGHT_TEMP=5.0(Equal weight)
    print("Loading latest optimized strategy weights...")
    weights_df = pd.read_pickle("weights.pkl")
    weights_df.index = pd.to_datetime(weights_df.index)

    TEST_START = pd.Timestamp("2020-01-01")

    # 準備乾淨的價格跟權重表對齊
    close_clean = (close.loc[TEST_START:]
                   .replace(0, np.nan)
                   .ffill().bfill()
                   .dropna(axis=1))

    weights_daily = weights_df.reindex(close.index, method="ffill").fillna(0.0)
    common = close_clean.columns.intersection(weights_daily.columns)
    c_vbt = close_clean[common]
    w_vbt = weights_daily.loc[TEST_START:, common]

    print(f"\n開始進行移動停損掃描 (Trailing Stop Loss)...")

    # TSL 參數掃描清單：從極嚴格(5%)到寬鬆(25%)，以及 0.0 (不停損)
    # vectorbt 對 tsl_stop 的吃法：0.15 代表從買入後高點回落 15% 觸發結算
    sweep_tsl = [0.0, 0.05, 0.10, 0.15, 0.18, 0.20, 0.25, 0.30]
    results = []

    for tsl in sweep_tsl:
        print(f"  Running Backtest with Target TSL = {tsl:.0%} ...")


        if tsl > 0:
            is_held = w_vbt > 0
            c_held = c_vbt.where(is_held, np.nan)
            s_c = c_held.unstack()
            s_hold = (~is_held).cumsum().unstack().astype(str) + "_" + s_c.index.get_level_values(0).astype(str)
            trailing_max = s_c.groupby(s_hold).cummax()
            dd = (trailing_max - s_c) / trailing_max
            trigger = dd >= tsl
            triggered = trigger.groupby(s_hold).cummax()
            w_tsl_s = w_vbt.unstack()
            w_tsl_s[triggered] = 0.0
            w_run = w_tsl_s.unstack(level=0)
        else:
            w_run = w_vbt

        pf = vbt.Portfolio.from_orders(
            close       = c_vbt,
            size        = w_run,
            size_type   = "targetpercent",
            group_by    = True,
            cash_sharing= True,
            fees        = 0.001425,
            slippage    = 0.001,
            init_cash   = 1_000_000,
        )

        stats = pf.stats()
        total_ret = stats["Total Return [%]"]
        max_dd = stats["Max Drawdown [%]"]
        monthly_ret = pf.value().resample("ME").last().pct_change().dropna()
        sharpe = float((monthly_ret.mean() / monthly_ret.std()) * (12**0.5))

        years = (pf.value().index[-1] - pf.value().index[0]).days / 365.25
        cagr = ((pf.value().iloc[-1] / pf.value().iloc[0]) ** (1/years)) - 1

        results.append({
            "TSL": f"{tsl*100:.0f}%" if tsl > 0 else "None",
            "CAGR": cagr,
            "Sharpe": sharpe,
            "Max DD": max_dd,
            "Total Return": total_ret
        })

finally:
    os.chdir(original_cwd)

Loading cached data from finmind_cache...
Loading latest optimized strategy weights...

開始進行移動停損掃描 (Trailing Stop Loss)...
  Running Backtest with Target TSL = 0% ...
  Running Backtest with Target TSL = 5% ...
  Running Backtest with Target TSL = 10% ...
  Running Backtest with Target TSL = 15% ...
  Running Backtest with Target TSL = 18% ...
  Running Backtest with Target TSL = 20% ...
  Running Backtest with Target TSL = 25% ...
  Running Backtest with Target TSL = 30% ...


In [2]:
# ── 整理結果並視覺化 ──
res_df = pd.DataFrame(results).set_index("TSL")
print("\n=== Trailing Stop Loss Sweep Results ===")
display(res_df.round(4))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(
    x=res_df.index, 
    y=res_df["CAGR"]*100, 
    name="CAGR (%)",
    marker_color="#9C27B0"
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=res_df.index, 
    y=abs(res_df["Max DD"]),
    name="|Max Drawdown| (%)",
    mode="lines+markers",
    line=dict(color="#FFF176", width=3)
), secondary_y=True)

fig.update_layout(
    title="CAGR vs Max Drawdown by Trailing Stop Loss Level",
    xaxis_title="Trailing Stop Loss Level (回落%停損)",
    template="plotly_dark", height=500
)
fig.update_yaxes(title_text="CAGR (%)", secondary_y=False)
fig.update_yaxes(title_text="Max Drawdown (%)", secondary_y=True)

fig.show()


=== Trailing Stop Loss Sweep Results ===


,CAGR,Sharpe,Max DD,Total Return
TSL,,,,
None,0.1083,0.5875,28.2953,88.5796
5%,0.0030,0.0801,28.5995,1.8720
10%,0.0152,0.1733,33.2866,9.7295
15%,0.0552,0.3849,28.7823,39.3064
18%,0.0664,0.4260,30.8429,48.7095
20%,0.0736,0.4550,27.9322,54.9271
25%,0.0820,0.4792,28.9016,62.6277
30%,0.0783,0.4547,28.8446,59.1953


## 專家見解

1. **如果 None (無停損) 表現反而最好**：
   代表這套策略「不需要個股停損保護」。因為在多頭市場中，很多動能飆股都是「洗盤再上」，如果只要回檔 15% 就把你洗下車，你會錯過後面 50% 的主升段！而我們本來就分散買了 40 檔，單一股票就算腰斬也只佔總資產不到 2%，根本傷不到筋骨。
   
2. **如果某個 TSL（例如 20%）的 Max DD 明顯下降且 CAGR 變高**：
   那就代表我們抓到了完美的「防摔氣墊」。

請觀察圖表與數據，若無停損反而賺比較多，我們就能理直氣壯的拒絕加入複雜的停損按鈕；如果加上停損效果更好，我們就在 `strategy.py` 把 TSL 參數加上去！